# Donor Pool Selection

## Objective

Synthetic control requires donor parks that represent plausible counterfactual outcomes for Rocky Mountain National Park.

This notebook creates three donor pools:

1. **Broad donor pool**: excludes treated parks only

2. **Comparable-access donor pool**: excludes parks with atypical access constraints or very low visitation

3. **Nearest-neighbor donor pool**: selected using pre-treatment visitation patterns

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

########################################################################
DATA_PATH = '../data/'
FIGURE_PATH = '../figures/'

RAW_FILE =  DATA_PATH+'raw/Main_Data.xls'
PARKS_FILE =  DATA_PATH+'raw/national_parks_lookup.csv'

VISITS_FILE =  DATA_PATH+'processed/monthly_visitation.csv'

INTERVENTION_FILE =  DATA_PATH+'processed/park_intervention_flags.csv'

### 1. Load the data 

In [2]:
visits = pd.read_csv(VISITS_FILE)

parks = pd.read_csv(INTERVENTION_FILE)

### 2. Create a broad donor pool 

In [3]:
parks["broad_donor"] = ((parks["donor_eligible"] == 1).astype(int))

In [4]:
parks

,park_name,park_code,intervention_type,start_year,timed_entry,donor_eligible,broad_donor
0,Pinnacles National Park,PINN,none,NaN,0,1,1
1,Dry Tortugas National Park,DRTO,none,NaN,0,1,1
2,Rocky Mountain National Park,ROMO,timed_entry,2020.0,1,0,0
3,Congaree National Park,CONG,none,NaN,0,1,1
4,Badlands National Park,BADL,none,NaN,0,1,1
...,...,...,...,...,...,...,...
57,Arches National Park,ARCH,timed_entry,2022.0,1,0,0
58,Isle Royale National Park,ISRO,none,NaN,0,1,1
59,Channel Islands National Park,CHIS,none,NaN,0,1,1
60,Lake Clark National Park,LACL,none,NaN,0,1,1


### 3. Identify atypical-access parks 

Exclude parks where visitation is driven by fundamentally different access constraints (fly-in, boat-only island, or remote Alaska access) because these are poor counteractuals for road-accessible park like RMNP. 

In [6]:
atypical_access = [

    # Remote Alaska parks
    "DENA",  # Denali
    "GAAR",  # Gates of the Arctic
    "KATM",  # Katmai
    "KEFJ",  # Kenai Fjords
    "KOVA",  # Kobuk Valley
    "LACL",  # Lake Clark
    "WRST",  # Wrangell-St. Elias
    "GLBA",  # Glacier Bay

    # Island / boat-access parks
    "ISRO",  # Isle Royale
    "DRTO",  # Dry Tortugas
    "NPSA",  # National Park of American Samoa
    "VIIS",  # Virgin Islands
    "CHIS",  # Channel Islands]
]

parks["atypical_access"] = (parks["park_code"].isin(atypical_access).astype(int))

### 4. Identify very low visitation parks 

In [7]:
annual_visits = (visits[visits["year"].between(2015,2019)].groupby(["park_code","year"])["visits"].sum().reset_index())

In [8]:
avg_visits = (annual_visits.groupby("park_code")["visits"].mean().reset_index(name="avg_annual_visits_2015_2019"))

avg_visits

,park_code,avg_annual_visits_2015_2019
0,ACAD,3319741.8
1,ARCH,1569450.4
2,BADL,1003976.4
3,BIBE,422847.2
4,BISC,529521.8
...,...,...
57,WICA,624249.2
58,WRST,76334.6
59,YELL,4121339.8
60,YOSE,4389654.4


In [9]:
parks = parks.merge(avg_visits, on="park_code", how="left")
parks

,park_name,park_code,intervention_type,start_year,timed_entry,donor_eligible,broad_donor,atypical_access,avg_annual_visits_2015_2019
0,Pinnacles National Park,PINN,none,NaN,0,1,1,0,210959.6
1,Dry Tortugas National Park,DRTO,none,NaN,0,1,1,1,66962.8
2,Rocky Mountain National Park,ROMO,timed_entry,2020.0,1,0,0,0,4474252.4
3,Congaree National Park,CONG,none,NaN,0,1,1,0,139265.0
4,Badlands National Park,BADL,none,NaN,0,1,1,0,1003976.4
...,...,...,...,...,...,...,...,...,...
57,Arches National Park,ARCH,timed_entry,2022.0,1,0,0,0,1569450.4
58,Isle Royale National Park,ISRO,none,NaN,0,1,1,1,24810.8
59,Channel Islands National Park,CHIS,none,NaN,0,1,1,1,369838.0
60,Lake Clark National Park,LACL,none,NaN,0,1,1,1,18662.2


### 5. Create comparable donor pool 

In [10]:
parks["low_visitation"] = (parks["avg_annual_visits_2015_2019"]< 100000).astype(int)

In [11]:
parks["comparable_donor"] = ((parks["broad_donor"] == 1) & (parks["atypical_access"] == 0) & (parks["low_visitation"] == 0)).astype(int)

In [12]:
parks[parks["comparable_donor"] == 1]["park_name"]

0                        Pinnacles National Park
3                         Congaree National Park
4                         Badlands National Park
5           Sequoia & Kings Canyon National Park
9                 Great Sand Dunes National Park
10                      Everglades National Park
11                       Wind Cave National Park
12                     Great Basin National Park
13                       Voyageurs National Park
15                    Gateway Arch National Park
16                     Yellowstone National Park
17    Black Canyon of the Gunnison National Park
19                         Olympic National Park
21                     Joshua Tree National Park
22                Petrified Forest National Park
23                        Big Bend National Park
24                     Canyonlands National Park
25                     Hot Springs National Park
26                 New River Gorge National Park
27             Guadalupe Mountains National Park
28                  

In [13]:
print ("Comparable Donor:", len(parks[parks["comparable_donor"] == 1]["park_name"]), "parks")

Comparable Donor: 41 parks


### 6. Create visitation features for nearest neighbors 

In [14]:
pre = visits[visits["year"].between(2015,2019)]

# feature 1: average visitors 
features = (pre.groupby("park_code")["visits"].mean().reset_index(name="avg_monthly_visits"))

# feature 2: seasonaility  -- june, july, august, september 
summer = (pre[pre["month"].isin([6,7,8,9])].groupby("park_code")["visits"].sum())

total = (pre.groupby("park_code")["visits"].sum())

seasonality = (summer / total).reset_index(name="summer_share")

# merge
features = features.merge(seasonality,on="park_code")

### 7: Nearest neighbor selection 

In [15]:
rmnp = features[features["park_code"] == "ROMO"]

donors = features[features["park_code"] != "ROMO"]

In [16]:
## Scale  
scaler = StandardScaler()

X = scaler.fit_transform(donors.drop(columns="park_code"))

target = scaler.transform(rmnp.drop(columns="park_code"))

### Fit 
nn = NearestNeighbors(n_neighbors=15)

nn.fit(X)

distances, indices = nn.kneighbors(target)

## get the parks 
nearest_codes = (donors.iloc[indices[0]]["park_code"].tolist())

print ("NN:", len(nearest_codes), "parks")

NN: 15 parks


In [17]:
## save the nearest neighbor flag 
parks["nearest_neighbor_donor"] = (parks["park_code"].isin(nearest_codes).astype(int))

### 8: Final check 

In [18]:
parks[["broad_donor","comparable_donor","nearest_neighbor_donor"]].sum()

broad_donor               55
comparable_donor          41
nearest_neighbor_donor    15
dtype: int64

### 9: Lets save the resulting file 

In [19]:
filename = 'donor_pool.csv' 
parks.to_csv("../data/processed/"+filename,index=False)